In [4]:
# ================================================
# CELL 1 — IMPORT & LOAD DATA
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
import os

warnings.filterwarnings('ignore')

# Load Dataset
df = pd.read_csv('../data/raw/Finance_Trends.csv')

print("Dataset berhasil dimuat!")
print(f"Shape awal: {df.shape}")
display(df.head())

Dataset berhasil dimuat!
Shape awal: (12000, 24)


,gender,age,Investment_Avenues,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold,...,Duration,Invest_Monitor,Expect,Avenue,What are your savings objectives?,Reason_Equity,Reason_Mutual,Reason_Bonds,Reason_FD,Source
0,Male,29,Yes,2,4,7,5,3,1,6,...,Less than 1 year,Weekly,20%-30%,Public Provident Fund,Health Care,Dividend,Fund Diversification,Assured Returns,Fixed Returns,Newspapers and Magazines
1,Male,28,Yes,2,3,6,5,1,4,7,...,Less than 1 year,Weekly,30%-40%,Public Provident Fund,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Risk Free,Television
2,Female,28,Yes,2,3,7,5,4,1,6,...,3-5 years,Daily,10%-20%,Mutual Fund,Health Care,Dividend,Better Returns,Assured Returns,Risk Free,Financial Consultants
3,Male,19,Yes,2,3,7,4,6,1,5,...,More than 5 years,Monthly,10%-20%,Equity,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Risk Free,Newspapers and Magazines
4,Male,32,Yes,2,4,7,5,3,1,6,...,More than 5 years,Weekly,10%-20%,Public Provident Fund,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Fixed Returns,Financial Consultants


In [5]:
# ================================================
# CELL 2 — SELEKSI FITUR
# ================================================

print("Kolom sebelum seleksi: ", list(df.columns))
print(f"Total kolom: {df.shape[1]}")

# Cek kolom yang dipertanyakan
print("\nCek Investment_Avenues:")
print(f"Total kolom: {df.shape[1]}")

print("\nCek Stock_Market:")
print(df['Investment_Avenues'].value_counts())

print("\nCek 'What are your saving objectives?':")
print(df['What are your savings objectives?'].value_counts())

Kolom sebelum seleksi:  ['gender', 'age', 'Investment_Avenues', 'Mutual_Funds', 'Equity_Market', 'Debentures', 'Government_Bonds', 'Fixed_Deposits', 'PPF', 'Gold', 'Stock_Marktet', 'Factor', 'Objective', 'Purpose', 'Duration', 'Invest_Monitor', 'Expect', 'Avenue', 'What are your savings objectives?', 'Reason_Equity', 'Reason_Mutual', 'Reason_Bonds', 'Reason_FD', 'Source']
Total kolom: 24

Cek Investment_Avenues:
Total kolom: 24

Cek Stock_Market:
Investment_Avenues
Yes    11082
No       918
Name: count, dtype: int64

Cek 'What are your saving objectives?':
What are your savings objectives?
Retirement Plan    7152
Health Care        3956
Education           892
Name: count, dtype: int64


In [6]:
# Terdapat 2 kolom yang dibuang
# Investment_Avenues → hampir semua "Yes", tidak informatif
# Stock_Marktet → redundan dengan kolom Equity_Market

kolom_buang = ['Investment_Avenues', 'Stock_Marktet']

df = df.drop(columns=kolom_buang)

# Rename kolom yang namanya aneh
df = df.rename(columns={
    'What are you savings objectives?': 'Savings_Objective'
})

print(f"✅ Kolom setelah seleksi: {df.shape[1]}")
print("Kolom tersisa:", list(df.columns))

✅ Kolom setelah seleksi: 22
Kolom tersisa: ['gender', 'age', 'Mutual_Funds', 'Equity_Market', 'Debentures', 'Government_Bonds', 'Fixed_Deposits', 'PPF', 'Gold', 'Factor', 'Objective', 'Purpose', 'Duration', 'Invest_Monitor', 'Expect', 'Avenue', 'What are your savings objectives?', 'Reason_Equity', 'Reason_Mutual', 'Reason_Bonds', 'Reason_FD', 'Source']


In [7]:
# ================================================
# CELL 3 — HANDLING INCONSISTENT VALUES
# ================================================

print("Cek nilai unik setiap kolom kategorikal:\n")

kolom_kategorikal = df.select_dtypes(include='object').columns.tolist()

for col in kolom_kategorikal:
    print(f"[{col}]")
    print(df[col].value_counts().to_dict())
    print()

Cek nilai unik setiap kolom kategorikal:

[gender]
{'Male': 7491, 'Female': 4509}

[Factor]
{'Returns': 7437, 'Risk': 4268, 'Locking Period': 295}

[Objective]
{'Capital Appreciation': 7789, 'Growth': 3341, 'Income': 870}

[Purpose]
{'Wealth Creation': 9614, 'Savings for Future': 1821, 'Returns': 565}

[Duration]
{'3-5 years': 3055, 'More than 5 years': 2985, 'Less than 1 year': 2981, '1-3 years': 2979}

[Invest_Monitor]
{'Daily': 4019, 'Weekly': 3995, 'Monthly': 3986}

[Expect]
{'30%-40%': 4117, '10%-20%': 3963, '20%-30%': 3920}

[Avenue]
{'Fixed Deposits': 3053, 'Mutual Fund': 3037, 'Public Provident Fund': 3001, 'Equity': 2909}

[What are your savings objectives?]
{'Retirement Plan': 7152, 'Health Care': 3956, 'Education': 892}

[Reason_Equity]
{'Capital Appreciation': 9028, 'Dividend': 2369, 'Liquidity': 603}

[Reason_Mutual]
{'Better Returns': 7250, 'Fund Diversification': 3899, 'Tax Benefits': 851}

[Reason_Bonds]
{'Assured Returns': 7883, 'Safe Investment': 3815, 'Tax Incentives

In [8]:
# Bersihkan Whitespace & Standarisasi Kapitalisasi

for col in kolom_kategorikal:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.title()

# Cek usia yang tidak wajar
print("Cek usia tidak wajar (dibawah 18 dan diatas 100):")
print(df[df['age'] < 18])
print(df[df['age'] > 100])

print("\n Data sudah dibersihkan!")
display(df.head())

Cek usia tidak wajar (dibawah 18 dan diatas 100):
Empty DataFrame
Columns: [gender, age, Mutual_Funds, Equity_Market, Debentures, Government_Bonds, Fixed_Deposits, PPF, Gold, Factor, Objective, Purpose, Duration, Invest_Monitor, Expect, Avenue, What are your savings objectives?, Reason_Equity, Reason_Mutual, Reason_Bonds, Reason_FD, Source]
Index: []

[0 rows x 22 columns]
Empty DataFrame
Columns: [gender, age, Mutual_Funds, Equity_Market, Debentures, Government_Bonds, Fixed_Deposits, PPF, Gold, Factor, Objective, Purpose, Duration, Invest_Monitor, Expect, Avenue, What are your savings objectives?, Reason_Equity, Reason_Mutual, Reason_Bonds, Reason_FD, Source]
Index: []

[0 rows x 22 columns]

 Data sudah dibersihkan!


,gender,age,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold,Factor,...,Duration,Invest_Monitor,Expect,Avenue,What are your savings objectives?,Reason_Equity,Reason_Mutual,Reason_Bonds,Reason_FD,Source
0,Male,29,2,4,7,5,3,1,6,Risk,...,Less Than 1 Year,Weekly,20%-30%,Public Provident Fund,Health Care,Dividend,Fund Diversification,Assured Returns,Fixed Returns,Newspapers And Magazines
1,Male,28,2,3,6,5,1,4,7,Returns,...,Less Than 1 Year,Weekly,30%-40%,Public Provident Fund,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Risk Free,Television
2,Female,28,2,3,7,5,4,1,6,Returns,...,3-5 Years,Daily,10%-20%,Mutual Fund,Health Care,Dividend,Better Returns,Assured Returns,Risk Free,Financial Consultants
3,Male,19,2,3,7,4,6,1,5,Risk,...,More Than 5 Years,Monthly,10%-20%,Equity,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Risk Free,Newspapers And Magazines
4,Male,32,2,4,7,5,3,1,6,Returns,...,More Than 5 Years,Weekly,10%-20%,Public Provident Fund,Retirement Plan,Capital Appreciation,Better Returns,Assured Returns,Fixed Returns,Financial Consultants


In [9]:
# ================================================
# CELL 4 — ENCODING KOLOM KATEGORIKAL
# ================================================

# Simpan salinan sebelum encoding
df_clean = df.copy()
df_encoded = df.copy()

# Pisahkan fitur dan target
TARGET = 'Avenue'
fitur_kategorikal = [col for col in df_encoded.select_dtypes(
    include='object').columns
    if col != TARGET]

print("Kolom yang di-encode:", fitur_kategorikal)
print("Target:", TARGET)

Kolom yang di-encode: ['gender', 'Factor', 'Objective', 'Purpose', 'Duration', 'Invest_Monitor', 'Expect', 'What are your savings objectives?', 'Reason_Equity', 'Reason_Mutual', 'Reason_Bonds', 'Reason_FD', 'Source']
Target: Avenue


In [10]:
# Label Encoding untuk semua kolom kategorikal
le_dict = {} # Simpan encoder untuk tiap kolom

for col in fitur_kategorikal:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col] = le
    print(f"✅ {col}: {list(le.classes_)} → {list(range(len(le.classes_)))}")

# Encoding TARGET secara terpisah
le_target = LabelEncoder()
df_encoded[TARGET] = le_target.fit_transform(df_encoded[TARGET])

print(f"\n✅ Target '{TARGET}' encoding:")
for i, cls in enumerate(le_target.classes_):
    print(f"   {cls} → {i}")    

✅ gender: ['Female', 'Male'] → [0, 1]
✅ Factor: ['Locking Period', 'Returns', 'Risk'] → [0, 1, 2]
✅ Objective: ['Capital Appreciation', 'Growth', 'Income'] → [0, 1, 2]
✅ Purpose: ['Returns', 'Savings For Future', 'Wealth Creation'] → [0, 1, 2]
✅ Duration: ['1-3 Years', '3-5 Years', 'Less Than 1 Year', 'More Than 5 Years'] → [0, 1, 2, 3]
✅ Invest_Monitor: ['Daily', 'Monthly', 'Weekly'] → [0, 1, 2]
✅ Expect: ['10%-20%', '20%-30%', '30%-40%'] → [0, 1, 2]
✅ What are your savings objectives?: ['Education', 'Health Care', 'Retirement Plan'] → [0, 1, 2]
✅ Reason_Equity: ['Capital Appreciation', 'Dividend', 'Liquidity'] → [0, 1, 2]
✅ Reason_Mutual: ['Better Returns', 'Fund Diversification', 'Tax Benefits'] → [0, 1, 2]
✅ Reason_Bonds: ['Assured Returns', 'Safe Investment', 'Tax Incentives'] → [0, 1, 2]
✅ Reason_FD: ['Fixed Returns', 'High Interest Rates', 'Risk Free'] → [0, 1, 2]
✅ Source: ['Financial Consultants', 'Internet', 'Newspapers And Magazines', 'Television'] → [0, 1, 2, 3]

✅ Target '

In [11]:
# Simpan mapping encoding untuk referensi
print("\n📋 MAPPING ENCODING TARGET:")
target_mapping = dict(zip(
    le_target.classes_,
    range(len(le_target.classes_))
))
print(target_mapping)

print("\nShape setelah encoding:", df_encoded.shape)
display(df_encoded.head())


📋 MAPPING ENCODING TARGET:
{'Equity': 0, 'Fixed Deposits': 1, 'Mutual Fund': 2, 'Public Provident Fund': 3}

Shape setelah encoding: (12000, 22)


,gender,age,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold,Factor,...,Duration,Invest_Monitor,Expect,Avenue,What are your savings objectives?,Reason_Equity,Reason_Mutual,Reason_Bonds,Reason_FD,Source
0,1,29,2,4,7,5,3,1,6,2,...,2,2,1,3,1,1,1,0,0,2
1,1,28,2,3,6,5,1,4,7,1,...,2,2,2,3,2,0,0,0,2,3
2,0,28,2,3,7,5,4,1,6,1,...,1,0,0,2,1,1,0,0,2,0
3,1,19,2,3,7,4,6,1,5,2,...,3,1,0,0,2,0,0,0,2,2
4,1,32,2,4,7,5,3,1,6,1,...,3,2,0,3,2,0,0,0,0,0


In [12]:
# ================================================
# CELL 5 — NORMALISASI DATA NUMERIK
# ================================================

kolom_numerik = ['age', 'Mutual_Funds', 'Equity_Market',
                 'Debentures', 'Government_Bonds',
                 'Fixed_Deposits', 'PPF', 'Gold']

print("Statistik SEBELUM scaling:")
display(df_encoded[kolom_numerik].describe())

# Terapkan StandardScaler
scaler = StandardScaler()
df_encoded[kolom_numerik] = scaler.fit_transform(
    df_encoded[kolom_numerik]
)

print("\nStatistik SETELAH scaling:")
display(df_encoded[kolom_numerik].describe())

print("\n✅ Normalisasi selesai!")

Statistik SEBELUM scaling:


,age,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,27.779000,2.534583,3.459000,5.777750,4.670333,3.556333,2.023583,5.978417
std,4.056316,1.168511,1.112245,1.638824,1.330276,1.757200,1.592086,1.119697
min,18.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000
25%,25.000000,2.000000,3.000000,5.000000,4.000000,2.000000,1.000000,6.000000
50%,28.000000,2.000000,4.000000,7.000000,5.000000,3.000000,1.000000,6.000000
75%,31.000000,3.000000,4.000000,7.000000,5.000000,5.000000,3.000000,7.000000
max,38.000000,7.000000,6.000000,7.000000,7.000000,7.000000,6.000000,7.000000



Statistik SETELAH scaling:


,age,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold
count,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,1.200000e+04,12000.000000,1.200000e+04
mean,3.671137e-17,-8.585725e-17,-4.500104e-17,-8.467301e-17,-2.877698e-16,-9.473903e-18,0.000000,-2.516506e-16
std,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042e+00,1.000042,1.000042e+00
min,-2.410909e+00,-1.313336e+00,-2.210936e+00,-2.915473e+00,-2.759192e+00,-1.454837e+00,-0.642946,-3.553266e+00
25%,-6.851330e-01,-4.575102e-01,-4.126961e-01,-4.745978e-01,-5.039265e-01,-8.857264e-01,-0.642946,1.927685e-02
50%,5.448521e-02,-4.575102e-01,4.864239e-01,7.458401e-01,2.478286e-01,-3.166154e-01,-0.642946,1.927685e-02
75%,7.941034e-01,3.983155e-01,4.864239e-01,7.458401e-01,2.478286e-01,8.216066e-01,0.613319,9.124126e-01
max,2.519879e+00,3.821618e+00,2.284664e+00,7.458401e-01,1.751339e+00,1.959829e+00,2.497718,9.124126e-01



✅ Normalisasi selesai!


In [13]:
# ================================================
# CELL 6 — SPLIT DATA TRAIN & TEST
# ================================================

# Pisahkan fitur (X) dan target (y)
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f"Shape X (fitur)  : {X.shape}")
print(f"Shape y (target) : {y.shape}")

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # ← pastikan distribusi label seimbang
)

print(f"\n✅ Hasil split:")
print(f"   Training set : {X_train.shape[0]:,} data ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"   Testing set  : {X_test.shape[0]:,} data ({X_test.shape[0]/len(X)*100:.0f}%)")

# Cek distribusi label di train & test
print(f"\nDistribusi TARGET di Train:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nDistribusi TARGET di Test:")
print(pd.Series(y_test).value_counts().sort_index())

Shape X (fitur)  : (12000, 21)
Shape y (target) : (12000,)

✅ Hasil split:
   Training set : 9,600 data (80%)
   Testing set  : 2,400 data (20%)

Distribusi TARGET di Train:
Avenue
0    2327
1    2442
2    2430
3    2401
Name: count, dtype: int64

Distribusi TARGET di Test:
Avenue
0    582
1    611
2    607
3    600
Name: count, dtype: int64


In [14]:
# ================================================
# CELL 7 — SIMPAN HASIL PREPROCESSING
# ================================================

# Pastikan folder processed ada
os.makedirs('../data/processed', exist_ok=True)

# 1. Simpan data yang sudah bersih (sebelum encoding)
df_clean.to_csv('../data/processed/cleaned_data.csv',
                index=False)

# 2. Simpan data yang sudah di-encode
df_encoded.to_csv('../data/processed/encoded_data.csv',
                  index=False)

# 3. Simpan train & test set
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("✅ Semua file berhasil disimpan!")
print("\nFile yang tersimpan di data/processed/:")
for f in os.listdir('../data/processed'):
    size = os.path.getsize(f'../data/processed/{f}')
    print(f"   📄 {f} ({size:,} bytes)")

✅ Semua file berhasil disimpan!

File yang tersimpan di data/processed/:
   📄 cleaned_data.csv (2,441,409 bytes)
   📄 encoded_data.csv (2,209,671 bytes)
   📄 X_test.csv (437,374 bytes)
   📄 X_train.csv (1,748,529 bytes)
   📄 y_test.csv (7,208 bytes)
   📄 y_train.csv (28,808 bytes)


In [15]:
# ================================================
# CELL 8 — RINGKASAN PREPROCESSING
# ================================================

print("=" * 60)
print("📋 RINGKASAN DATA PREPARATION")
print("=" * 60)
print(f"""
📊 DATASET:
   • Shape awal      : (12000, 24)
   • Shape akhir     : {df_encoded.shape}
   • Kolom dibuang   : Investment_Avenues, Stock_Marktet

🔤 ENCODING:
   • Kolom kategorikal yang di-encode : {len(fitur_kategorikal)}
   • Metode                           : Label Encoding
   • Target mapping                   : {target_mapping}

📏 NORMALISASI:
   • Kolom yang di-scale  : {len(kolom_numerik)}
   • Metode               : StandardScaler

✂️  SPLIT DATA:
   • Training set  : {X_train.shape[0]:,} data (80%)
   • Testing set   : {X_test.shape[0]:,} data (20%)
   • Stratified    : Yes (distribusi seimbang)

💾 OUTPUT:
   • cleaned_data.csv
   • encoded_data.csv
   • X_train.csv, X_test.csv
   • y_train.csv, y_test.csv
""")
print("=" * 60)
print("✅ DATA PREPARATION SELESAI!")
print("   Siap lanjut ke tahap MODELING! 🚀")
print("=" * 60)

📋 RINGKASAN DATA PREPARATION

📊 DATASET:
   • Shape awal      : (12000, 24)
   • Shape akhir     : (12000, 22)
   • Kolom dibuang   : Investment_Avenues, Stock_Marktet

🔤 ENCODING:
   • Kolom kategorikal yang di-encode : 13
   • Metode                           : Label Encoding
   • Target mapping                   : {'Equity': 0, 'Fixed Deposits': 1, 'Mutual Fund': 2, 'Public Provident Fund': 3}

📏 NORMALISASI:
   • Kolom yang di-scale  : 8
   • Metode               : StandardScaler

✂️  SPLIT DATA:
   • Training set  : 9,600 data (80%)
   • Testing set   : 2,400 data (20%)
   • Stratified    : Yes (distribusi seimbang)

💾 OUTPUT:
   • cleaned_data.csv
   • encoded_data.csv
   • X_train.csv, X_test.csv
   • y_train.csv, y_test.csv

✅ DATA PREPARATION SELESAI!
   Siap lanjut ke tahap MODELING! 🚀


In [1]:
# ================================================
# CELL 9 — FINAL CHECK DATA PREPARATION
# ================================================

import os
import pandas as pd

print("=" * 55)
print("🔍 FINAL CHECK — DATA PREPARATION")
print("=" * 55)

# -----------------------------------------------
# CHECK 1 — File CSV sudah tersimpan?
# -----------------------------------------------
print("\n CHECK 1: File di data/processed/")
print("-" * 40)

files_expected = [
    'cleaned_data.csv',
    'encoded_data.csv',
    'X_train.csv',
    'X_test.csv',
    'y_train.csv',
    'y_test.csv'
]

semua_ada = True
for f in files_expected:
    path = f'../data/processed/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"✅ {f} ({size:,} bytes)")
    else:
        print(f"❌ {f} — TIDAK ADA!")
        semua_ada = False

if semua_ada:
    print("\n✅ Semua file tersimpan dengan benar!")
else:
    print("\n⚠️ Ada file yang belum tersimpan!")

# -----------------------------------------------
# CHECK 2 — Shape data sudah benar?
# -----------------------------------------------
print("\n📊 CHECK 2: Shape Data")
print("-" * 40)

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

total = len(X_train) + len(X_test)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")
print(f"\nProporsi Train : {len(X_train)/total*100:.1f}%")
print(f"Proporsi Test  : {len(X_test)/total*100:.1f}%")

if len(X_train)/total >= 0.79:
    print("✅ Proporsi split sudah benar!")
else:
    print("⚠️ Proporsi split tidak sesuai!")

# -----------------------------------------------
# CHECK 3 — Tidak ada NaN?
# -----------------------------------------------
print("\n🔎 CHECK 3: Missing Values")
print("-" * 40)

nan_train = X_train.isnull().sum().sum()
nan_test  = X_test.isnull().sum().sum()

print(f"NaN di X_train : {nan_train}")
print(f"NaN di X_test  : {nan_test}")


if nan_train == 0 and nan_test == 0:
    print("✅ Tidak ada missing values!")
else:
    print("⚠️ Masih ada missing values!")

# -----------------------------------------------
# CHECK 4 — Semua kolom numerik?
# -----------------------------------------------
print("\n🔢 CHECK 4: Tipe Data")
print("-" * 40)

tipe_unik = X_train.dtypes.unique()
print(f"Tipe data X_train: {tipe_unik}")

ada_object = any(str(t) == 'object' for t in tipe_unik)
if not ada_object:
    print("✅ Semua kolom sudah numerik!")
else:
    print("⚠️ Masih ada kolom bertipe teks (object)!")

# -----------------------------------------------
# CHECK 5 — Distribusi label seimbang?
# -----------------------------------------------
print("\n⚖️  CHECK 5: Distribusi Label")
print("-" * 40)

print("Distribusi y_train:")
print(y_train.value_counts().sort_index())

print("\nDistribusi y_test:")
print(y_test.value_counts().sort_index())

pct_min = y_train.value_counts(normalize=True).min() * 100
pct_max = y_train.value_counts(normalize=True).max() * 100

if (pct_max - pct_min) < 10:
    print("\n✅ Distribusi label seimbang!")
else:
    print("\n⚠️ Distribusi label tidak seimbang!")

# -----------------------------------------------
# KESIMPULAN FINAL CHECK
# -----------------------------------------------
print("\n" + "=" * 55)
print("📋 KESIMPULAN FINAL CHECK")
print("=" * 55)
print(f"""
□ File tersimpan     : {'✅' if semua_ada else '❌'}
□ Shape data benar   : {'✅' if len(X_train)/total >= 0.79 else '❌'}
□ Tidak ada NaN      : {'✅' if nan_train == 0 else '❌'}
□ Semua numerik      : {'✅' if not ada_object else '❌'}
□ Label seimbang     : {'✅' if (pct_max - pct_min) < 10 else '❌'}
""")

if semua_ada and nan_train == 0 and not ada_object:
    print("🚀 DATA PREPARATION SELESAI!")
    print("   Siap lanjut ke tahap MODELING!")
else:
    print("⚠️  Ada yang perlu diperbaiki dulu!")

print("=" * 55)

🔍 FINAL CHECK — DATA PREPARATION

 CHECK 1: File di data/processed/
----------------------------------------
✅ cleaned_data.csv (2,441,409 bytes)
✅ encoded_data.csv (2,209,671 bytes)
✅ X_train.csv (1,748,529 bytes)
✅ X_test.csv (437,374 bytes)
✅ y_train.csv (28,808 bytes)
✅ y_test.csv (7,208 bytes)

✅ Semua file tersimpan dengan benar!

📊 CHECK 2: Shape Data
----------------------------------------
X_train : (9600, 21)
X_test  : (2400, 21)
y_train : (9600,)
y_test  : (2400,)

Proporsi Train : 80.0%
Proporsi Test  : 20.0%
✅ Proporsi split sudah benar!

🔎 CHECK 3: Missing Values
----------------------------------------
NaN di X_train : 0
NaN di X_test  : 0
✅ Tidak ada missing values!

🔢 CHECK 4: Tipe Data
----------------------------------------
Tipe data X_train: [dtype('int64') dtype('float64')]
✅ Semua kolom sudah numerik!

⚖️  CHECK 5: Distribusi Label
----------------------------------------
Distribusi y_train:
Avenue
0    2327
1    2442
2    2430
3    2401
Name: count, dtype: int64